# MTL Archives dataset-led opportunity scan

**TL;DR.** The archive is strongest in aerial urban change, waterfront, infrastructure, public space, and industrial transformation. The best first evidence packet is therefore a district or site-system activation, led by the Old Port footprint and tested against SDC Vieux-Montréal, not a hotel selected in advance.

## Context and method

This notebook profiles the local research artifacts as they existed on September 1, 2026. It keeps the 13,499-row canonical/scored corpus separate from the 14,822-row VLM and taxonomy source grain and from the 18,462-identity reconciled graph. Candidate scores are reviewed hypotheses. They do not represent buyer intent.

In [1]:
from pathlib import Path
from collections import Counter
import csv, json, os, statistics

data_root = Path(os.environ.get('MTL_ARCHIVES_DATA_ROOT', '../mtl-archives-search/data/mtl_archives')).expanduser().resolve()
assert data_root.exists(), f'Missing data root: {data_root}'
print(f'Data root: {data_root}')

Data root: /Users/wiel/Development/mtl-archives-search/data/mtl_archives


In [2]:
def read_jsonl(path):
    with path.open() as handle:
        for line in handle:
            if line.strip():
                yield json.loads(line)

def first_present(row, *keys):
    for key in keys:
        value = row.get(key)
        if value not in (None, '', [], {}):
            return value
    return None

## Corpus grains and enrichment coverage

In [3]:
scored = list(read_jsonl(data_root / 'manifest_scored.jsonl'))
vlm_path = data_root / 'reports/autoresearch_vlm_full/manifest_vlm_structured_full_detailed_llava7b.jsonl'
vlm = list(read_jsonl(vlm_path))
taxonomy = list(read_jsonl(data_root / 'reports/autoresearch_taxonomy/taxonomy_labels.jsonl'))
geocoded = list(read_jsonl(data_root / 'manifest_geocoded.jsonl'))

coord_rows = [r for r in geocoded if first_present(r, 'latitude', 'lat') is not None and first_present(r, 'longitude', 'lon', 'lng') is not None]
print({
    'canonical_scored_rows': len(scored),
    'canonical_unique_ids': len({r.get('metadata_filename') or r.get('record_link_id') for r in scored}),
    'vlm_source_rows': len(vlm),
    'taxonomy_source_rows': len(taxonomy),
    'coordinate_bearing_rows': len(coord_rows),
    'coordinate_coverage_pct_of_14822': round(100 * len(coord_rows) / max(len(geocoded), 1), 2),
})

{'canonical_scored_rows': 13499, 'canonical_unique_ids': 13499, 'vlm_source_rows': 14822, 'taxonomy_source_rows': 14822, 'coordinate_bearing_rows': 162, 'coordinate_coverage_pct_of_14822': 1.09}


In [4]:
theme_counts = Counter()
category_counts = Counter()
vantage_counts = Counter()
for row in taxonomy:
    category_counts.update([first_present(row, 'primaryCategory', 'primary_category', 'category') or 'unknown'])
    taxonomy_fields = row.get('taxonomy') or {}
    vantage_counts.update([taxonomy_fields.get('vantage') or first_present(row, 'vantage') or 'unknown'])
    themes = taxonomy_fields.get('themes') or row.get('themes') or row.get('theme_labels') or []
    if isinstance(themes, str):
        themes = [themes]
    theme_counts.update(themes)
print('Top primary categories:', category_counts.most_common(10))
print('Vantage:', vantage_counts.most_common())
print('Top themes:', theme_counts.most_common(12))

Top primary categories: [('aerial_general', 8373), ('aerial_waterfront', 3407), ('aerial_residential', 1803), ('aerial_industrial', 511), ('uncertain', 452), ('document_map', 191), ('ground_photo', 57), ('people_event', 13), ('street_commercial', 7), ('ground_transit', 5)]
Vantage: [('aerial', 14175), ('unknown', 452), ('document_or_map', 110), ('ground', 85)]
Top themes: [('park_green_space', 6020), ('waterfront', 3583), ('residential', 2920), ('transit', 1619), ('industrial', 1296), ('construction', 848), ('winter', 350), ('crowd_event', 310), ('civic_institutional', 73), ('commercial', 53)]


## OCR quality boundary

In [5]:
ocr = list(read_jsonl(data_root / 'manifest_ocr.jsonl'))
confidences = []
errors = 0
for row in ocr:
    conf = first_present(row, 'ocr_confidence', 'confidence')
    if isinstance(conf, (int, float)):
        confidences.append(float(conf))
    if row.get('ocr_error') or row.get('error'):
        errors += 1
print({
    'rows': len(ocr),
    'unique_ids': len({r.get('metadata_filename') or r.get('id') for r in ocr}),
    'errors': errors,
    'mean_confidence': round(statistics.mean(confidences), 4),
    'median_confidence': round(statistics.median(confidences), 4),
    'at_least_0_6': sum(c >= 0.6 for c in confidences),
    'at_least_0_7': sum(c >= 0.7 for c in confidences),
})

{'rows': 13568, 'unique_ids': 13499, 'errors': 15, 'mean_confidence': 0.2776, 'median_confidence': 0.2662, 'at_least_0_6': 315, 'at_least_0_7': 155}


## Candidate score validation

In [6]:
score_path = Path('candidate-scores-v1.csv').resolve()
with score_path.open() as handle:
    scores = list(csv.DictReader(handle))
for row in scores:
    archive = float(row['archive_fit_100'])
    market = float(row['market_fit_100'])
    expected = 2 * archive * market / (archive + market)
    actual = float(row['balanced_fit_100'])
    assert abs(expected - actual) < 0.06, (row['candidate'], expected, actual)
print([(r['rank'], r['candidate'], r['balanced_fit_100'], r['gate']) for r in scores])

[('1', 'Old Port of Montréal Corporation', '73.3', 'Explore only'), ('2', 'SDC Vieux-Montréal', '73.0', 'Explore only'), ('3', 'SDC Montréal centre-ville', '66.3', 'Explore only'), ('4', 'PHI Contemporary', '62.0', 'Explore only'), ('5', 'Tourisme Montréal', '62.0', 'Explore only'), ('6', 'GI Quo Vadis', '61.7', 'Explore only'), ('7', 'Gray Collection', '61.7', 'Explore only'), ('8', 'Hôtel Nelligan reference concept', '30.2', 'Hold')]


## Takeaways

1. The dataset has a right to exist in district-scale urban transformation and heritage interpretation before it has a right to exist in hotel decor.
2. Old Port Corporation and SDC Vieux-Montréal form the first tier, but their rank is provisional because exact place linkage and access routes are incomplete.
3. The next artifact should be a reviewed 6–12 record evidence packet with canonical identities, unique visual families, claim labels, rights boundaries, and a Google-search baseline.
4. No outreach is justified by this analysis alone.